In [5]:
import numpy as np
from scipy import stats

np.random.seed(42)


def z_r(sample: np.ndarray) -> float:
    return 0.5 * (np.min(sample) + np.max(sample))


def z_q(sample: np.ndarray) -> float:
    return 0.5 * (np.quantile(sample, 0.25) + np.quantile(sample, 0.75))


def z_tr(sample: np.ndarray, gamma: float = 0.1) -> float:
    n = len(sample)
    k = int(np.floor(gamma * n))
    if k == 0:
        return float(np.mean(sample))
    return float(np.mean(np.sort(sample)[k:n - k]))


def run_simulation(generator, n: int, m: int = 1000) -> dict:
    names = ["mean", "median", "z_R", "z_Q", "z_tr"]
    values = {name: [] for name in names}
    for _ in range(m):
        s = generator(n)
        values["mean"].append(float(np.mean(s)))
        values["median"].append(float(np.median(s)))
        values["z_R"].append(z_r(s))
        values["z_Q"].append(z_q(s))
        values["z_tr"].append(z_tr(s))
    result = {}
    for name in names:
        arr = np.array(values[name])
        ez = float(np.mean(arr))
        dz = float(np.mean(arr ** 2) - ez ** 2)
        result[name] = {"E": ez, "D": dz}
    return result


def gen_normal(n):  return np.random.normal(0, 1, n)
def gen_cauchy(n):  return stats.cauchy.rvs(0, 1, size=n)
def gen_laplace(n): return np.random.laplace(0, 1 / np.sqrt(2), n)
def gen_poisson(n): return np.random.poisson(5, n)
def gen_uniform(n): return np.random.uniform(-np.sqrt(3), np.sqrt(3), n)


DISTRIBUTIONS = {
    "Normal N(0,1)":          gen_normal,
    "Cauchy C(0,1)":          gen_cauchy,
    "Laplace L(0,1/sqrt2)":   gen_laplace,
    "Poisson P(5)":           gen_poisson,
    "Uniform U(-sqrt3,sqrt3)": gen_uniform,
}

NS = [10, 100, 1000]
CHARS = ["mean", "median", "z_R", "z_Q", "z_tr"]
LABELS = {"mean": "x̄", "median": "med", "z_R": "z_R", "z_Q": "z_Q", "z_tr": "z_tr"}


def main():
    for dist_name, gen in DISTRIBUTIONS.items():
        print(f"\n{'='*72}")
        print(f"  {dist_name}")
        print(f"{'='*72}")
        header = f"{'Char':10}  {'n=10':>22}  {'n=100':>22}  {'n=1000':>22}"
        print(header)
        print("-" * 72)
        results = {n: run_simulation(gen, n) for n in NS}
        for c in CHARS:
            row = f"{LABELS[c]:10}"
            for n in NS:
                e = results[n][c]['E']
                d = results[n][c]['D']
                cell = f"{e:.2f} ± {d:.2f}"
                row += f"  {cell:>22}"
            print(row)


if __name__ == "__main__":
    main()


  Normal N(0,1)
Char                          n=10                   n=100                  n=1000
------------------------------------------------------------------------
x̄                    -0.00 ± 0.10             0.00 ± 0.01            -0.00 ± 0.00
med                   -0.01 ± 0.13             0.00 ± 0.01            -0.00 ± 0.00
z_R                   -0.00 ± 0.18             0.00 ± 0.09            -0.01 ± 0.06
z_Q                   -0.00 ± 0.11             0.00 ± 0.01            -0.00 ± 0.00
z_tr                  -0.00 ± 0.10             0.00 ± 0.01            -0.00 ± 0.00

  Cauchy C(0,1)
Char                          n=10                   n=100                  n=1000
------------------------------------------------------------------------
x̄                  0.47 ± 3013.63        -7.39 ± 72113.00           0.19 ± 526.77
med                   -0.01 ± 0.37            -0.01 ± 0.02            -0.00 ± 0.00
z_R                2.40 ± 75124.41  -371.12 ± 179885162.64    86.87 ± 123